In [ ]:
import pandas as pd;
print(pd.__version__)

In [ ]:
import os, time, json, getpass, requests, pandas as pd
from pathlib import Path

# Make paths work from anywhere: after this, 'data/raw/...' always means <project>/data/raw/...
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)
os.makedirs('data/raw', exist_ok=True)
print('Working in project folder:', ROOT.name)

# ListenBrainz requires a way to contact you. USE YOUR REAL EMAIL.
YOUR_EMAIL = getpass.getpass('Contact email for the User-Agent (hidden): ')
assert '@' in YOUR_EMAIL or YOUR_EMAIL.startswith('http'), 'Enter a real email or URL'

BASE = 'https://api.listenbrainz.org'
HEADERS = {'User-Agent': f'ListeningAnalysisProject/1.0 ( {YOUR_EMAIL} )'}

# Optional token (higher rate limits). In VS Code the prompt appears at the TOP of the window.
token = getpass.getpass('ListenBrainz token (press Enter to skip): ')
if token:
    HEADERS['Authorization'] = f'Token {token}'

def get_json(path, params=None):
    for attempt in range(5):
        r = requests.get(BASE + path, params=params, headers=HEADERS, timeout=30)
        if r.status_code == 429:                  # rate limited: wait as the server says
            wait = int(r.headers.get('X-RateLimit-Reset-In', 10))
            print(f'Rate limited, waiting {wait}s')
            time.sleep(wait + 1)
            continue
        if r.status_code == 404:                  # user not found
            return None
        r.raise_for_status()                      # any other error: stop and show it
        if int(r.headers.get('X-RateLimit-Remaining', 99)) <= 1:
            time.sleep(int(r.headers.get('X-RateLimit-Reset-In', 1)) + 1)
        time.sleep(1.1)                           # never more than one call per second
        return r.json()
    raise RuntimeError('Too many retries')

In [ ]:
good = Path('data/usernames.txt').read_text().split()
print(len(good), 'users loaded')

In [ ]:
def find_usernames(obj):
    """Collect every 'user_name' value, whatever shape the response has."""
    names = []
    if isinstance(obj, dict):
        if 'user_name' in obj:
            names.append(obj['user_name'])
        for v in obj.values():
            names += find_usernames(v)
    elif isinstance(obj, list):
        for v in obj:
            names += find_usernames(v)
    return names

terms = ['music', 'rock', 'jazz', 'indie', 'beats', 'dj']
candidates = []
for t in terms:
    data = get_json('/1/search/users/', {'search_term': t})
    found = find_usernames(data) if data else []
    print(t, '->', len(found), 'users')
    candidates += found

candidates = list(dict.fromkeys(candidates))   # remove duplicates, keep order
print(len(candidates), 'unique candidates')
print(candidates[:10])

In [ ]:
MIN_LISTENS = 3000
TARGET_USERS = 12

good = []
for i, u in enumerate(candidates):
    d = get_json(f'/1/user/{u}/listen-count')
    n = d['payload']['count'] if d else 0
    print(f'candidate {i}: {n} listens')          # prints the number, never the name
    if n >= MIN_LISTENS:
        good.append(u)
    if len(good) >= TARGET_USERS:
        break

print(len(good), 'users selected')
Path('data/usernames.txt').write_text('\n'.join(good))

In [ ]:
sample = get_json(f'/1/user/{good[0]}/listens', {'count': 3})
for l in sample['payload']['listens']:
    l = dict(l)
    l.pop('user_name', None)                  # keep the name out of the saved output
    print(json.dumps(l, indent=2)[:1500])
    print('-----')

In [ ]:
def fetch_listens(username, max_listens=5000, page_size=1000):
    rows, max_ts = [], None
    while len(rows) < max_listens:
        params = {'count': page_size}
        if max_ts:
            params['max_ts'] = max_ts            # only listens older than this
        data = get_json(f'/1/user/{username}/listens', params)
        if data is None:
            print('a user was not found')
            break
        listens = data['payload']['listens']
        if not listens:                           # nothing older: history exhausted
            break
        for l in listens:
            tm = l.get('track_metadata') or {}
            info = tm.get('additional_info') or {}
            rows.append({
                'user': username,
                'listened_at': l.get('listened_at'),
                'track': tm.get('track_name'),
                'artist': tm.get('artist_name'),
                'album': tm.get('release_name'),
                'duration_ms': info.get('duration_ms'),
                'music_service': info.get('music_service'),
                'client': info.get('submission_client'),
            })
        max_ts = listens[-1]['listened_at']       # the oldest listen on this page
        if len(listens) < page_size:              # a short page means the last page
            break
    return pd.DataFrame(rows).head(max_listens)

In [ ]:
test = fetch_listens(good[0], max_listens=2000)
print(test.shape)
print(test.isna().mean().round(2))
test.drop(columns='user').head()              # drop the username column before displaying

In [ ]:
frames = []
for i, u in enumerate(good):
    path = Path(f'data/raw/{u}.csv')
    if path.exists():                             # already downloaded: reuse it
        frames.append(pd.read_csv(path))
        print(f'user {i + 1}: loaded from cache')
        continue
    f = fetch_listens(u, max_listens=5000)
    f.to_csv(path, index=False)
    frames.append(f)
    print(f'user {i + 1}: {len(f)} listens downloaded')

raw = pd.concat(frames, ignore_index=True)
raw.to_csv('data/raw/listens_raw.csv', index=False)
print(raw.shape)

In [ ]:
labels = {u: f'user_{i + 1}' for i, u in enumerate(good)}

print(raw.isna().mean().round(3))
span = raw.groupby('user')['listened_at'].agg(['count', 'min', 'max'])
span['first_listen'] = pd.to_datetime(span['min'], unit='s')
span['last_listen'] = pd.to_datetime(span['max'], unit='s')
span.index = span.index.map(labels)              # anonymous labels for display
display(span[['count', 'first_listen', 'last_listen']])

print()
print(f"{len(raw):,} listens from {raw['user'].nunique()} users")
print('Earliest listen:', pd.to_datetime(raw['listened_at'].min(), unit='s'))
print('Latest listen:  ', pd.to_datetime(raw['listened_at'].max(), unit='s'))

In [ ]:
raw = pd.read_csv('data/raw/listens_raw.csv')
start_rows = len(raw)
df = raw.copy()

# 1. Remove rows missing the essentials
df = df.dropna(subset=['track', 'artist', 'listened_at'])
print('After removing nulls:', len(df))

# 2. Convert Unix seconds to a real datetime (UTC, timezone removed)
df['ts'] = pd.to_datetime(df['listened_at'], unit='s', utc=True).dt.tz_localize(None)

# 3. Tidy text so 'Radiohead ' and 'Radiohead' match
df['artist'] = df['artist'].str.strip()
df['track'] = df['track'].str.strip()

# 4. Remove exact duplicates (pagination can overlap by one row)
df = df.drop_duplicates(subset=['user', 'ts', 'track', 'artist'])
print('After removing duplicates:', len(df))

# 5. Anonymize users BEFORE anything gets published
mapping = {u: f'user_{i+1}' for i, u in enumerate(df['user'].unique())}
df['user'] = df['user'].map(mapping)

# 6. Features for analysis
df['year'] = df['ts'].dt.year
df['month'] = df['ts'].dt.month
df['day_of_week'] = df['ts'].dt.day_name()
df['is_weekend'] = df['ts'].dt.dayofweek >= 5

print(f'Started with {start_rows}, ended with {len(df)}')

In [ ]:
df.to_csv('data/clean_listens.csv', index=False)
print('Cleaned data saved to data/clean_listens.csv')

right now, each row repeats the artist and the album text. this is kinda awkward and real DB's store each fact once and link tables with IDs. Below is gonna be the creation of a star scheme, we will have 1 central fact table (aka listens, one row per event), with multiple dim tables (users, artists, tracks)

In [ ]:
import sqlite3

users = df[['user']].drop_duplicates().reset_index(drop=True)
users['user_id'] = users.index + 1

artists = df[['artist']].drop_duplicates().reset_index(drop=True)
artists['artist_id'] = artists.index + 1

tracks = (df[['track', 'artist', 'album']]
          .drop_duplicates(subset=['track', 'artist'])
          .reset_index(drop=True))
tracks['track_id'] = tracks.index + 1
tracks = tracks.merge(artists, on='artist')[['track_id', 'track', 'album', 'artist_id']]

listens = (df.merge(users, on='user')
             .merge(artists, on='artist')
             .merge(tracks[['track_id', 'track', 'artist_id']], on=['track', 'artist_id'])
             [['ts', 'user_id', 'track_id', 'year', 'month', 'day_of_week', 'is_weekend']])

print(len(df), len(listens))   # these MUST match

In [ ]:
conn = sqlite3.connect('data/listening.db')
users.to_sql('users', conn, if_exists='replace', index=False)
artists.to_sql('artists', conn, if_exists='replace', index=False)
tracks.to_sql('tracks', conn, if_exists='replace', index=False)
listens['ts'] = listens['ts'].astype(str)
listens.to_sql('listens', conn, if_exists='replace', index=False)

# Indexes make joins fast, and employers like seeing them
conn.execute('CREATE INDEX IF NOT EXISTS idx_listens_user ON listens(user_id)')
conn.execute('CREATE INDEX IF NOT EXISTS idx_listens_track ON listens(track_id)')
conn.execute('CREATE INDEX IF NOT EXISTS idx_tracks_artist ON tracks(artist_id)')
conn.commit()

print(pd.read_sql('SELECT COUNT(*) AS n FROM listens', conn))

used to find the total number of users and the date rang

In [ ]:
print(f"{len(df):,} listens")
print(df['user'].nunique(), 'users')
print('Earliest:', df['ts'].min())
print('Latest:  ', df['ts'].max())

used for running my SQL queries.
comment out those you dont want ran

In [ ]:
def run(path):
    return pd.read_sql(open(path).read(), conn)

run('sql/01_top_artists.sql')


In [ ]:
def run(path):
    return pd.read_sql(open(path).read(), conn)

run('sql/02_top_artist_per_user.sql')

In [ ]:
def run(path):
    return pd.read_sql(open(path).read(), conn)

run('sql/03_most_shared_artists.sql')

In [ ]:
def run(path):
    return pd.read_sql(open(path).read(), conn)

run('sql/04_listening_concentration.sql')

In [ ]:
def run(path):
    return pd.read_sql(open(path).read(), conn)

run('sql/05_monthly_trends.sql')

In [ ]:
def run(path):
    return pd.read_sql(open(path).read(), conn)

run('sql/06_weekend_share.sql')

In [ ]:
def run(path):
    return pd.read_sql(open(path).read(), conn)

run('sql/test stuff.sql')

In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns

os.makedirs('images', exist_ok=True)       # the charts get saved here
print(os.getcwd())
print(sorted(os.listdir('sql')))           # should list your saved query files

Chart 1: most-listened artists

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

top = run('sql/01_top_artists.sql')
plt.figure(figsize=(8, 5))
sns.barplot(data=top, y='artist', x='listens', color='steelblue')
plt.title('Top 10 artists across all users')
plt.xlabel('Listens')
plt.ylabel('')
plt.tight_layout()
plt.savefig('images/top_artists.png', dpi=150)
plt.show()

The 10 most-played artists make up 13.6% of the 28,114 listens, but 5 of them were played by a single user and 0 by more than 3. Bilmuri leads with 996 plays (3.5%), all from 1 listener, so total plays here reflect individual heavy fans rather than shared taste.

Chart 2: monthly listening trends
I dropped the first and last months because, from what I see, they are partial

In [ ]:
m = run('sql/05_monthly_trends.sql').iloc[1:-1]
m['period'] = pd.to_datetime(m[['year', 'month']].assign(day=1))
plt.figure(figsize=(10, 4))
plt.plot(m['period'], m['n'])
plt.title('Total listens per month')
plt.ylabel('Listens')
plt.tight_layout()
plt.savefig('images/monthly_trend.png', dpi=150)
plt.show()

Each user’s data covers their most recent 5,000 listens, so older months include fewer users than recent months. Total monthly listens therefore reflect how many users contribute data as much as listening behavior. The chart below shows listens per contributing user to adjust for this.

Chart 3: how concentrated a users taste is

In [ ]:
c = run('sql/04_listening_concentration.sql')
plt.figure(figsize=(8, 4))
sns.barplot(data=c, x='user_id', y='top10_share_pct', color='teal')
plt.title('Share of plays from each user\'s top 10 artists')
plt.xlabel('User')
plt.ylabel('Percent of plays')
plt.tight_layout()
plt.savefig('images/concentration.png', dpi=150)
plt.show()

In [ ]:
c = run('sql/04_listening_concentration.sql')
print(c['top10_share_pct'].describe())


Listening concentration varies widely in this sample of 6 users: the top 10 artists account for between 9.7% and 59.7% of each user’s plays (mean 27.25%). One user gets 59.7% of their plays from just 10 artists, while another spreads listening across many more.